In [1]:
def generate_pcx(table=None,max_x=3,save_path=None):
    from sklearn.decomposition import PCA
    import pandas as pd
    pca_full = pd.DataFrame(PCA().fit_transform(table))
    for i in range(1,max_x,1):
        X=pca_full.iloc[:,:i]
        X.to_csv(save_path+'/'+str(i)+'.csv')
    return

In [2]:
def random_split(input_data=None, output_data=None,size=0.2):
    '''
    train_test_split using a random selection
    '''
    from sklearn.model_selection import train_test_split
    X,x,Y,y = train_test_split(input_data, output_data, test_size=size, random_state=137)
    return X,x,Y,y

In [3]:
def ks_split(input_data, output_data,size=0.2):
    '''
    train_test_split using the kennard_stone method
    '''
    from kennard_stone import train_test_split
    X,x,Y,y = train_test_split(input_data, output_data, test_size=size,random_state=137)
    return X,x,Y,y

In [4]:
def normalization_of_input(table):
    '''
    Input normalization function
    '''
    from tensorflow.keras.layers import Normalization
    normalizer = Normalization()
    normalizer.adapt(table)
    return normalizer(A).numpy()

In [5]:
def separate_compounds_from_metalic_component(folder_to_save=None, folder_with_cifs=None):
    ''' 
    To perform the chemically relevant splitting and the elemental leave-one-out (LOO) analysis, the MOFs must be grouped according to their metallic element.
    This function copies all CIF files and organizes them into folders based on the 2nd–4th period metal they contain.
    If a MOF contains more than one such metal, it is placed in the mix category.
    '''
    import os
    import shutil
    from glob import glob
    from ase import io
    from natsort import natsorted

    # Non-metals
    common_elements = {'H', 'C', 'Cl', 'Br', 'N', 'O', 'F', 'B', 'P', 'S'}

    # Base destination path
    base_dest = folder_to_save

    # Metals of 2-4 periods
    target_elements = {'Al', 'Ca', 'Co', 'Cr', 'Cu', 'Fe', 'Ga', 'Ge', 'K', 'Li', 'Mg', 'Mn','Na', 'Ni', 'Sc', 'Ti', 'V', 'Zn'}

    # Creates subfolders inside specified folder_to_save
    for elem in target_elements.union({'mix'}):
        os.makedirs(os.path.join(base_dest, elem), exist_ok=True)

    # Scans for target_elemnets
    for filepath in natsorted(glob(folder_with_cifs)):
        try:
            cif = io.read(filepath, index=0)
            elements = set(cif.get_chemical_symbols()) - common_elements
        except Exception:
            continue

        if len(elements) == 1:
            elem = elements.pop()
            if elem in target_elements:
                dest = os.path.join(base_dest, elem)
            else:
                dest = os.path.join(base_dest, 'mix')
        else:
            dest = os.path.join(base_dest, 'mix')

        shutil.copy2(filepath, dest)
    return

In [6]:
def return_elemental_holdout(number, path):
    '''
    Performs the elemental hold-out technique.
    Returns X_train and Y_train containing MOFs with all metallic elements except the specified one.
    X_test and Y_test contain MOFs with the held-out element.
    
    For this method to work, within a single folder (the path folder):

    The representation arrays (e.g., SM) of all MOFs containing a given element must be saved as NumPy arrays, named after the metallic element they contain.

    Their corresponding properties must be stored in CSV files, also named after their metallic composition.
    '''
    from natsort import natsorted
    import glob
    import numpy as np
    import pandas as pd
    ele_ho_data={}
    ele_ho_prop={}
    q=0
    for i in natsorted(glob.glob(path+'/*.npy')):
        ele_ho_data[q]=np.load(i)
        ele_ho_prop[q]=pd.read_csv(i.split('.')[0]+'.csv')
        q=q+1
    x_test=ele_ho_data.pop(number)
    y_test=ele_ho_prop.pop(number)
    x_train=np.concatenate(([ele_ho_data[x] for x in list(ele_ho_data)]),axis=0)
    y_train=pd.concat([ele_ho_prop[x] for x in list(ele_ho_prop)],axis=0)
    return (x_train,x_test,y_train,y_test)